In [2]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
import joblib

df = pd.read_csv("../data/cleaned_spotify.csv")

print(df.shape)
df.head()

(89740, 20)


,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,1,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,"acoustic, j-pop, singer-songwriter, songwriter"
1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,1,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,"acoustic, chill"
2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,0,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,0,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,2,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


In [3]:
features = [
    "danceability",
    "energy",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo"
]

X = df[features].copy()

X.head()

,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo
0,0.676,0.4610,-6.746,0.1430,0.0322,0.000001,0.3580,0.715,87.917
1,0.420,0.1660,-17.235,0.0763,0.9240,0.000006,0.1010,0.267,77.489
2,0.438,0.3590,-9.734,0.0557,0.2100,0.000000,0.1170,0.120,76.332
3,0.266,0.0596,-18.515,0.0363,0.9050,0.000071,0.1320,0.143,181.740
4,0.618,0.4430,-9.681,0.0526,0.4690,0.000000,0.0829,0.167,119.949


In [4]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

print(X_scaled.shape)

(89740, 9)


In [5]:
model = NearestNeighbors(
    metric="cosine",
    algorithm="brute"
)

model.fit(X_scaled)

print("Recommendation model trained successfully!")

Recommendation model trained successfully!


In [6]:
def recommend_songs(track_name, artist=None, n_recommendations=10):
    
    matches = df[
        df["track_name"].str.lower() == track_name.lower()
    ]

    if artist is not None:
        matches = matches[
            matches["artists"].str.lower().str.contains(
                artist.lower(),
                regex=False
            )
        ]

    if matches.empty:
        return f"Song '{track_name}' was not found."

    song_index = matches.index[0]

    song_features = X_scaled[song_index].reshape(1, -1)

    distances, indices = model.kneighbors(
        song_features,
        n_neighbors=n_recommendations + 1
    )

    recommendations = df.iloc[indices[0][1:]].copy()

    recommendations["similarity_score"] = (
        1 - distances[0][1:]
    )

    return recommendations[
        [
            "track_name",
            "artists",
            "track_genre",
            "popularity",
            "similarity_score"
        ]
    ].reset_index(drop=True)

In [7]:
recommend_songs(
    "Can't Help Falling in Love",
    "Kina Grannis",
    10
)

/Users/abdur-rahman/Projects/music-recommendation-system/.venv/lib/python3.12/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/abdur-rahman/Projects/music-recommendation-system/.venv/lib/python3.12/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/abdur-rahman/Projects/music-recommendation-system/.venv/lib/python3.12/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


,track_name,artists,track_genre,popularity,similarity_score
0,Repetitious (Zamaangard),Baaroot,iranian,7,0.994545
1,Uzun Gecələr,Rashid Beibutov,romance,11,0.992960
2,把歌談心,林姍姍;David Ling Jr,cantopop,27,0.992578
3,In A Sentimental Mood,Ella Fitzgerald,"blues, jazz",0,0.991857
4,Chalana,Almir Sater,mpb,41,0.991500
5,Ozhidanyie - Zachem Sidish Polunchi,Ivan Kozlovsky,romance,3,0.991165
6,Жалобно стонет,Соня Тимофеева,romance,0,0.990541
7,Gulabi,Sushant KC,indie-pop,45,0.990048
8,Глядя на луч пурпурного заката,Валерий Агафонов,romance,0,0.988515
9,"Gianni Schicchi: ""O mio babbino caro""",Giacomo Puccini;Mirella Freni;Orchestra del Ma...,opera,24,0.988261


In [8]:
joblib.dump(model, "../models/nearest_neighbors.joblib")
joblib.dump(scaler, "../models/scaler.joblib")

print("Model and scaler saved successfully!")

Model and scaler saved successfully!


In [9]:
print("NaN values:", np.isnan(X_scaled).sum())
print("Infinite values:", np.isinf(X_scaled).sum())
print("Largest absolute value:", np.abs(X_scaled).max())

NaN values: 0
Infinite values: 0
Largest absolute value: 7.8582967135863315


In [10]:
from sklearn.preprocessing import MultiLabelBinarizer
from scipy.sparse import csr_matrix, hstack

genre_lists = df["track_genre"].str.split(", ")

mlb = MultiLabelBinarizer(sparse_output=True)

genre_matrix = mlb.fit_transform(genre_lists)

print("Number of genre features:", len(mlb.classes_))
print("Genre matrix shape:", genre_matrix.shape)

Number of genre features: 114
Genre matrix shape: (89740, 114)


In [11]:
audio_weight = 1.0
genre_weight = 1.5

audio_matrix = csr_matrix(X_scaled * audio_weight)
weighted_genre_matrix = genre_matrix * genre_weight

X_hybrid = hstack([
    audio_matrix,
    weighted_genre_matrix
]).tocsr()

print("Hybrid feature matrix:", X_hybrid.shape)

Hybrid feature matrix: (89740, 123)


In [12]:
hybrid_model = NearestNeighbors(
    metric="cosine",
    algorithm="brute",
    n_jobs=-1
)

hybrid_model.fit(X_hybrid)

print("Hybrid recommendation model ready!")

Hybrid recommendation model ready!


In [18]:
def recommend_songs_v2(track_name, artist=None, n_recommendations=10):

    matches = df[
        df["track_name"].str.lower() == track_name.lower()
    ]

    if artist is not None:
        matches = matches[
            matches["artists"].str.lower().str.contains(
                artist.lower(),
                regex=False
            )
        ]

    if matches.empty:
        return f"Song '{track_name}' was not found."

    song_index = matches.index[0]
    input_track = df.loc[song_index]

    song_vector = X_hybrid[song_index]

    # Ask for extra neighbours because some will be filtered out
    distances, indices = hybrid_model.kneighbors(
        song_vector,
        n_neighbors=n_recommendations + 30
    )

    recommendations = df.iloc[indices[0]].copy()
    recommendations["similarity_score"] = 1 - distances[0]

    # Remove the exact input track
    recommendations = recommendations[
        recommendations["track_id"] != input_track["track_id"]
    ]

    # Remove tracks with exactly the same title
    recommendations = recommendations[
        recommendations["track_name"].str.lower()
        != input_track["track_name"].lower()
    ]

    recommendations = recommendations.head(n_recommendations)

    return recommendations[
        [
            "track_name",
            "artists",
            "track_genre",
            "popularity",
            "similarity_score"
        ]
    ].reset_index(drop=True)

In [19]:
recommend_songs_v2(
    "Can't Help Falling in Love",
    "Kina Grannis",
    10
)

,track_name,artists,track_genre,popularity,similarity_score
0,Can't Help Falling In Love - Piano Version,Kina Grannis,acoustic,54,0.985743
1,I Ain't Ever Loved No One - Acoustic,Donovan Woods;Tenille Townes,acoustic,57,0.977292
2,Wish List,Canyon City,acoustic,46,0.964693
3,Poison & Wine,The Civil Wars,acoustic,54,0.963641
4,"See You Again, Love Me Like You Do, Sugar - Ac...",Megan Davies,acoustic,54,0.956596
5,太陽さん,Ichiko Aoba,acoustic,55,0.956433
6,Beautiful Disaster,Chord Overstreet,acoustic,41,0.954453
7,Already Mine,Us The Duo,acoustic,46,0.954324
8,The Will To Death,John Frusciante,acoustic,51,0.951673
9,So Are You To Me,Eastmountainsouth,acoustic,52,0.949503


# Model Evaluation

Since the dataset does not contain explicit user ratings or listening histories,
recommendation quality is evaluated using genre consistency.

The baseline model uses only numerical audio features, while the hybrid model
combines audio features with genre information.

In [20]:
def genre_set(genre_string):
    return set(genre_string.split(", "))


def has_genre_overlap(input_genres, recommended_genres):
    return len(
        genre_set(input_genres) &
        genre_set(recommended_genres)
    ) > 0

In [21]:
def evaluate_v1(sample_size=500, k=10, random_state=42):

    sample = df.sample(
        sample_size,
        random_state=random_state
    )

    total_matches = 0
    total_recommendations = 0

    for idx in sample.index:

        distances, indices = model.kneighbors(
            X_scaled[idx].reshape(1, -1),
            n_neighbors=k + 1
        )

        input_genres = df.loc[idx, "track_genre"]

        for rec_idx in indices[0][1:]:

            if has_genre_overlap(
                input_genres,
                df.loc[rec_idx, "track_genre"]
            ):
                total_matches += 1

            total_recommendations += 1

    return total_matches / total_recommendations

In [22]:
def evaluate_v2(sample_size=500, k=10, random_state=42):

    sample = df.sample(
        sample_size,
        random_state=random_state
    )

    total_matches = 0
    total_recommendations = 0

    for idx in sample.index:

        distances, indices = hybrid_model.kneighbors(
            X_hybrid[idx],
            n_neighbors=k + 1
        )

        input_genres = df.loc[idx, "track_genre"]

        for rec_idx in indices[0][1:]:

            if has_genre_overlap(
                input_genres,
                df.loc[rec_idx, "track_genre"]
            ):
                total_matches += 1

            total_recommendations += 1

    return total_matches / total_recommendations

In [23]:
v1_genre_score = evaluate_v1()
v2_genre_score = evaluate_v2()

print(f"V1 Genre Consistency@10: {v1_genre_score:.2%}")
print(f"V2 Genre Consistency@10: {v2_genre_score:.2%}")

/Users/abdur-rahman/Projects/music-recommendation-system/.venv/lib/python3.12/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/abdur-rahman/Projects/music-recommendation-system/.venv/lib/python3.12/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/abdur-rahman/Projects/music-recommendation-system/.venv/lib/python3.12/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/abdur-rahman/Projects/music-recommendation-system/.venv/lib/python3.12/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/abdur-rahman/Projects/music-recommendation-system/.venv/lib/python3.12/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/abdur-rahman/Projects/music-recommendation-system/.venv/lib/p

V1 Genre Consistency@10: 17.64%
V2 Genre Consistency@10: 98.26%


In [24]:
print("NaN values in hybrid matrix:", np.isnan(X_hybrid.data).sum())
print("Infinite values in hybrid matrix:", np.isinf(X_hybrid.data).sum())
print("Hybrid matrix shape:", X_hybrid.shape)

NaN values in hybrid matrix: 0
Infinite values in hybrid matrix: 0
Hybrid matrix shape: (89740, 123)


In [25]:
joblib.dump(hybrid_model, "../models/hybrid_recommender.joblib")
joblib.dump(scaler, "../models/scaler.joblib")
joblib.dump(mlb, "../models/genre_encoder.joblib")

print("Final recommendation model saved successfully!")

Final recommendation model saved successfully!


In [26]:
evaluation_results = pd.DataFrame({
    "model": [
        "Audio-only baseline",
        "Audio + genre hybrid"
    ],
    "genre_consistency_at_10": [
        v1_genre_score,
        v2_genre_score
    ]
})

evaluation_results

,model,genre_consistency_at_10
0,Audio-only baseline,0.1764
1,Audio + genre hybrid,0.9826


In [27]:
evaluation_results.to_csv(
    "../models/evaluation_results.csv",
    index=False
)

print("Evaluation results saved!")

Evaluation results saved!
